# 01 - AI Business Intelligence: Dataset Exploration

এই notebook-এ Store Sales এবং Online Retail dataset লোড করে data quality, schema, missing values, duplicate rows, date coverage এবং basic preprocessing পরীক্ষা করা হবে। এখানে কোনো AI provider বা database connection দরকার নেই। বড় CSV-র ক্ষেত্রে inspection sample ব্যবহার হবে, তবে পুরো CSV-র row count memory-efficient ভাবে গণনা করা হবে।

In [2]:
from pathlib import Path
import pandas as pd
from IPython.display import display

DATA_FILES = [
    'train.csv', 'test.csv', 'stores.csv', 'transactions.csv',
    'holidays_events.csv', 'oil.csv', 'sample_submission.csv',
    'Online Retail.xlsx',
]

def find_data_dir():
    roots = [Path.cwd(), *Path.cwd().parents]
    candidates = []
    for root in roots:
        candidates.extend([root / 'dataset', root / 'notebooks' / 'dataset', root / 'ai-service' / 'notebooks'])
    for candidate in candidates:
        if any((candidate / name).is_file() for name in DATA_FILES):
            return candidate
    raise FileNotFoundError(
        'Dataset folder not found. Expected files in notebooks/dataset. '
        'Move or copy the datasets there, then rerun this cell.'
    )

DATA_DIR = find_data_dir()
print(f'Dataset folder: {DATA_DIR}')

Dataset folder: e:\9th semi\AI Project\ai-service\notebooks\dataset


## File inventory and dataset profile

`train.csv`, `test.csv`, and `sample_submission.csv` are capped to a preview sample for profiling. The full row count is counted by iterating over each CSV, not by loading the whole file.

In [3]:
LARGE_CSVS = {'train.csv', 'test.csv', 'sample_submission.csv'}
PREVIEW_ROWS = 100_000
datasets = {}
inventory = []

def count_csv_rows(path):
    with path.open('r', encoding='utf-8-sig', errors='replace') as handle:
        return max(sum(1 for _ in handle) - 1, 0)

for name in DATA_FILES:
    path = DATA_DIR / name
    if not path.is_file():
        print(f'MISSING: {name}')
        continue
    size_mb = path.stat().st_size / (1024 ** 2)
    if path.suffix.lower() == '.csv':
        full_rows = count_csv_rows(path)
        row_limit = PREVIEW_ROWS if name in LARGE_CSVS else None
        frame = pd.read_csv(path, nrows=row_limit, low_memory=False)
        sampled = row_limit is not None and full_rows > len(frame)
    else:
        frame = pd.read_excel(path, nrows=PREVIEW_ROWS)
        full_rows = None
        sampled = len(frame) == PREVIEW_ROWS
    datasets[name] = frame
    inventory.append({
        'file': name,
        'file_size_mb': round(size_mb, 2),
        'full_csv_rows': full_rows,
        'loaded_rows': len(frame),
        'sampled': sampled,
        'columns': len(frame.columns),
    })

display(pd.DataFrame(inventory))

,file,file_size_mb,full_csv_rows,loaded_rows,sampled,columns
0,train.csv,116.16,3000888.0,100000,True,6
1,test.csv,0.97,28512.0,28512,False,5
2,stores.csv,0.00,54.0,54,False,5
3,transactions.csv,1.48,83488.0,83488,False,3
4,holidays_events.csv,0.02,350.0,350,False,6
5,oil.csv,0.02,1218.0,1218,False,2
6,sample_submission.csv,0.33,28512.0,28512,False,2
7,Online Retail.xlsx,22.62,NaN,100000,True,8


In [4]:
def profile_frame(name, frame):
    print(f'\n### {name}')
    print(f'Loaded shape: {frame.shape}')
    print(f'Duplicate rows in loaded data: {frame.duplicated().sum():,}')
    profile = pd.DataFrame({
        'dtype': frame.dtypes.astype(str),
        'missing': frame.isna().sum(),
        'missing_pct': (frame.isna().mean() * 100).round(2),
        'unique_values': frame.nunique(dropna=True),
    })
    display(profile)
    date_columns = [
        column for column in frame.columns
        if any(token in column.lower() for token in ('date', 'time'))
    ]
    for column in date_columns:
        parsed = pd.to_datetime(frame[column], errors='coerce')
        if parsed.notna().any():
            print(f'{column} range: {parsed.min()} to {parsed.max()} '
                  f'({parsed.isna().sum():,} unparsed/missing in loaded data)')
    display(frame.head(5))

for name, frame in datasets.items():
    profile_frame(name, frame)


### train.csv
Loaded shape: (100000, 6)
Duplicate rows in loaded data: 0


,dtype,missing,missing_pct,unique_values
id,int64,0,0.0,100000
date,str,0,0.0,57
store_nbr,int64,0,0.0,54
family,str,0,0.0,33
sales,float64,0,0.0,15007
onpromotion,int64,0,0.0,1


date range: 2013-01-01 00:00:00 to 2013-02-26 00:00:00 (0 unparsed/missing in loaded data)


,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0



### test.csv
Loaded shape: (28512, 5)
Duplicate rows in loaded data: 0


,dtype,missing,missing_pct,unique_values
id,int64,0,0.0,28512
date,str,0,0.0,16
store_nbr,int64,0,0.0,54
family,str,0,0.0,33
onpromotion,int64,0,0.0,212


... (20 outputs left out)


### Online Retail.xlsx
Loaded shape: (100000, 8)
Duplicate rows in loaded data: 924


,dtype,missing,missing_pct,unique_values
InvoiceNo,object,0,0.00,4657
StockCode,object,0,0.00,3143
Description,str,299,0.30,3049
Quantity,int64,0,0.00,299
InvoiceDate,datetime64[us],0,0.00,4207
UnitPrice,float64,0,0.00,572
CustomerID,float64,34915,34.92,1664
Country,str,0,0.00,29


InvoiceDate range: 2010-12-01 08:26:00 to 2011-02-23 13:37:00 (0 unparsed/missing in loaded data)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Cleaning and preprocessing checks

These examples create cleaned copies for inspection. Missing values and returned/cancelled Online Retail transactions are reported rather than silently removed.

In [5]:
if 'train.csv' in datasets:
    train_preview = datasets['train.csv'].copy()
    if 'date' in train_preview:
        train_preview['date'] = pd.to_datetime(train_preview['date'], errors='coerce')
    if 'family' in train_preview:
        train_preview['family'] = train_preview['family'].astype('string').str.strip()
    print('Store Sales preview:')
    print('Invalid/missing dates:', int(train_preview['date'].isna().sum())
          if 'date' in train_preview else 'date column unavailable')
    print('Missing values by column:')
    display(train_preview.isna().sum().rename('missing').to_frame())
    print('Rows before/after dropping only invalid dates:')
    if 'date' in train_preview:
        cleaned_train_preview = train_preview.dropna(subset=['date']).copy()
        print(len(train_preview), '->', len(cleaned_train_preview))

if 'Online Retail.xlsx' in datasets:
    retail_preview = datasets['Online Retail.xlsx'].copy()
    if 'InvoiceDate' in retail_preview:
        retail_preview['InvoiceDate'] = pd.to_datetime(
            retail_preview['InvoiceDate'], errors='coerce'
        )
    required = {'Quantity', 'UnitPrice'}
    if required.issubset(retail_preview.columns):
        retail_preview['LineRevenue'] = (
            pd.to_numeric(retail_preview['Quantity'], errors='coerce')
            * pd.to_numeric(retail_preview['UnitPrice'], errors='coerce')
        )
        print('Online Retail preview:')
        print('Missing values by column:')
        display(retail_preview.isna().sum().rename('missing').to_frame())
        print('Negative-quantity return/cancellation rows in preview:',
              int((retail_preview['Quantity'] < 0).sum()))
        print('Negative/zero prices in preview:',
              int((retail_preview['UnitPrice'] <= 0).sum()))
        display(retail_preview[['Quantity', 'UnitPrice', 'LineRevenue']].describe())
    else:
        print('Online Retail revenue check skipped: Quantity/UnitPrice columns missing.')

Store Sales preview:
Invalid/missing dates: 0
Missing values by column:


,missing
id,0
date,0
store_nbr,0
family,0
sales,0
onpromotion,0


Rows before/after dropping only invalid dates:
100000 -> 100000
Online Retail preview:
Missing values by column:


,missing
InvoiceNo,0
StockCode,0
Description,299
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,34915
Country,0
LineRevenue,0


Negative-quantity return/cancellation rows in preview: 2079
Negative/zero prices in preview: 482


,Quantity,UnitPrice,LineRevenue
count,100000.00000,100000.000000,100000.000000
mean,8.89292,5.396608,17.277084
std,336.56270,120.891000,372.387334
min,-74215.00000,0.000000,-77183.600000
25%,1.00000,1.250000,3.360000
50%,3.00000,2.510000,8.420000
75%,8.00000,4.250000,16.980000
max,74215.00000,16888.020000,77183.600000


## Findings to record

- Confirm every expected file is present and review the reported row counts.
- Record the date coverage and missing columns before choosing analyses.
- Store Sales provides sales quantities/values and store geography; it does not contain a country field or unit price.
- Online Retail supports product, country, quantity, and revenue analysis using `Quantity * UnitPrice`.
- Keep returns/cancellations visible and decide explicitly how each business metric should treat them.